# LEVIR-Ship — ONLY new P3/P4/P5 + FTSC YAMLs for YOLOv9t and YOLO11n

Two arms only: **V9T_P345_FTSC** and **V11N_P345_FTSC**, each at seeds 42/43/44 (**six runs maximum**). This is a completion of the P345+FTSC corner for these two YOLO families, **not a full 2×2 factorial rerun**. The original v8 factorial notebook is superseded in this file; the other 10 configurations are NOT dispatched.

Preserved from `current_live_training_notebok_LEV.ipynb`: LEVIR dataset download/validation, GitHub code retrieval, native Ultralytics weight acquisition, fork dataloader workers, HF authentication and completion-marker-last archival. No token is stored in the notebook source.

Run offline checks and preflights first. Enable training only intentionally. The notebook **checks Hugging Face first**, restores already completed seed runs, trains only missing valid cells and uploads each completed seed immediately. Any partial HF archive or network failure blocks automatic retraining.

In [ ]:
from pathlib import Path
import os, sys, json
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")

ROOT = Path("/marimo/yolo_code")
GITHUB_REPO_URL = "https://github.com/TLMHoang-AI/YOLO_FTSC.git"
GITHUB_BRANCH = "main"
HF_DATA_REPO = "duyle2408/LevirShipData"
HF_RESULTS_REPO = "TLMHoang/FTSC_results"

SYNC_GITHUB_CODE = True             # safe Git fast-forward only
INSTALL_MISSING_DEPENDENCIES = False
PROMPT_FOR_HF_TOKEN = False         # opt-in secure runtime entry
PREPARE_DATASET = True
PREPARE_PRETRAINED = True
RUN_OFFLINE_TESTS = True
RUN_STATIC_PREFLIGHT = True
RUN_MODEL_PREFLIGHT = False         # manual smoke test; automatically runs before training
RUN_HF_PREFLIGHT_AND_RESTORE = False
RUN_TRAINING = False                # explicit opt-in; only 2 new P345+FTSC YAMLs
RUN_AGGREGATION = False             # only after all 6 seed runs complete
ALLOW_RESUME = False

DEVICE = "cuda:0"
WORKERS = 4
CASES = ["V9T_P345_FTSC", "V11N_P345_FTSC"]
SEEDS = [42, 43, 44]
assert CASES and len(CASES) == len(set(CASES)) and set(CASES).issubset({"V9T_P345_FTSC", "V11N_P345_FTSC"})
assert set(SEEDS).issubset({42, 43, 44}) and len(SEEDS) == len(set(SEEDS))
OUTPUT_ROOT = ROOT / "runs/levir_ftsc_p345_v9_v11_v1"
HF_CACHE = Path("/marimo/hf_results_cache/levir_ftsc_p345_v9_v11_v1")
if PROMPT_FOR_HF_TOKEN and not os.environ.get("HF_TOKEN", "").strip():
    from getpass import getpass
    os.environ["HF_TOKEN"] = getpass("Hugging Face access token: ").strip()
HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "").strip()
print("HF token configured:", bool(HF_TOKEN), "| GitHub token configured:", bool(GITHUB_TOKEN))
print("GPU training enabled:", RUN_TRAINING, "| only selected:", CASES, "| seeds:", SEEDS)

In [ ]:
if INSTALL_MISSING_DEPENDENCIES:
    import subprocess
    packages = ["huggingface_hub", "pyyaml", "pycocotools", "ultralytics-thop",
                "albumentations", "kornia", "timm", "opencv-python-headless", "pillow"]
    subprocess.run([sys.executable, "-m", "pip", "install", *packages], check=True)
else:
    import yaml
    from huggingface_hub import HfApi, snapshot_download
    print("Found Python YAML and Hugging Face clients. Dependency install skipped.")

In [ ]:
# Safe adaptation of source notebook's GitHub checkout: no destructive reset, no token in URL.
if SYNC_GITHUB_CODE:
    import subprocess, base64
    git_env = os.environ.copy()
    git_env["GIT_TERMINAL_PROMPT"] = "0"
    if GITHUB_TOKEN:
        basic = base64.b64encode(("x-access-token:" + GITHUB_TOKEN).encode()).decode()
        git_env["GIT_CONFIG_COUNT"] = "1"
        git_env["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"
        git_env["GIT_CONFIG_VALUE_0"] = "AUTHORIZATION: basic " + basic
    def git_checked(arguments, cwd=None):
        result = subprocess.run(["git", *arguments], cwd=cwd, env=git_env,
                                text=True, capture_output=True)
        if result.returncode:
            # Never show credential-bearing subprocess environments.
            raise RuntimeError("GitHub checkout failed. Check repository permissions, "
                               "branch, local modifications and GITHUB_TOKEN.")
        return result.stdout.strip()
    if not ROOT.exists():
        ROOT.parent.mkdir(parents=True, exist_ok=True)
        git_checked(["clone", "--branch", GITHUB_BRANCH, "--single-branch",
                     GITHUB_REPO_URL, str(ROOT)])
    elif (ROOT / ".git").is_dir():
        git_checked(["fetch", "origin", GITHUB_BRANCH], cwd=ROOT)
        git_checked(["merge", "--ff-only", f"origin/{GITHUB_BRANCH}"], cwd=ROOT)
    else:
        raise RuntimeError(f"{ROOT} exists but is not a Git repository; refusing to overwrite it.")
    print("YOLO FTSC checkout revision:", git_checked(["rev-parse", "--short", "HEAD"], cwd=ROOT))
else:
    print("GitHub sync disabled; using existing local checkout:", ROOT)
assert ROOT.is_dir(), f"Code checkout missing: {ROOT}"

In [ ]:
VENDOR = (ROOT / "models_related/ultralytics").resolve()
required_source = [
    "train_levir_scripts/train_levir_ftsc_p345_v9_v11_v1.py",
    "train_levir_scripts/ftsc_p345_v9_v11_hf_v1.py",
    "train_levir_scripts/test_levir_ftsc_p345_v9_v11_v1.py",
    "models_related/models_config/yolov9/levir/yolov9t_levir_p345_ftsc_factorial.yaml",
    "models_related/models_config/yolo11/levir/yolo11n_levir_p345_ftsc_factorial.yaml",
    "utils/ftsc_results_uploader.py",
]
missing = [rel for rel in required_source if not (ROOT / rel).is_file()]
if missing:
    raise RuntimeError("P345 v9/v11 code missing in Live checkout. Push your updated "
                       "local yolo_code to GitHub first: " + str(missing))
for item in (str(ROOT), str(VENDOR)):
    if item not in sys.path:
        sys.path.insert(0, item)
import ultralytics
if VENDOR not in Path(ultralytics.__file__).resolve().parents:
    raise RuntimeError("Wrong Ultralytics package imported. Restart kernel to load FTSC fork.")
from train_levir_scripts import train_levir_ftsc_p345_v9_v11_v1 as suite
from train_levir_scripts import ftsc_p345_v9_v11_hf_v1 as hf_suite
from train_levir_scripts import train_all_levir_yolov8n_p2_routing as workflow
assert set(suite.CASES) == set(CASES), "Runner case set drift"
print("Verified two-case-only YOLOv9t + YOLO11n source and local Ultralytics fork")

In [ ]:
# Offline regression tests do not train or write to HF.
if RUN_OFFLINE_TESTS:
    import unittest
    tests = unittest.defaultTestLoader.loadTestsFromNames([
        "train_levir_scripts.test_levir_ftsc_p345_v9_v11_v1",
    ])
    outcome = unittest.TextTestRunner(verbosity=2).run(tests)
    if not outcome.wasSuccessful():
        raise RuntimeError("P345 FTSC regression tests failed. Training blocked.")
    print("Offline P345 v9/v11 and HF tests passed.")

In [ ]:
# Faithful LEVIR dataset retrieval: original notebook checks 3,896 matching pairs.
from huggingface_hub import snapshot_download

def find_data_root():
    candidates = [Path.cwd(), Path.cwd() / "LevirShipData",
                  Path("/marimo"), Path("/marimo/LevirShipData"),
                  ROOT / "LevirShipData"]
    for folder in dict.fromkeys(candidates):
        if not ((folder / "All Images").is_dir()
                and (folder / "All Annotations").is_dir()):
            continue
        images = {p.stem for p in (folder / "All Images").glob("*.png")}
        labels = {p.stem for p in (folder / "All Annotations").glob("*.txt")}
        if len(images) == len(labels) == 3896 and images == labels:
            return folder.resolve()
    return None

DATA_ROOT = find_data_root()
if PREPARE_DATASET and DATA_ROOT is None:
    if not HF_TOKEN:
        raise RuntimeError("Set HF_TOKEN in the Marimo Live environment; missing LEVIR dataset.")
    print("Downloading LEVIR dataset from", HF_DATA_REPO)
    snapshot_download(repo_id=HF_DATA_REPO, repo_type="dataset",
                      token=HF_TOKEN, local_dir=str(Path("/marimo")), max_workers=2)
    DATA_ROOT = find_data_root()
if PREPARE_DATASET:
    assert DATA_ROOT is not None, "LEVIR dataset download incomplete: expected 3,896 image/label pairs"
    from argparse import Namespace
    DATASET_ROOT = ROOT / "datasets"
    data_yaml = DATASET_ROOT / "levir_ship_yolo_seed42/levir_ship.yaml"
    if data_yaml.is_file():
        workflow.validate_split(data_yaml, DATA_ROOT)  # reject changes instead of silently rewriting
    else:
        data_yaml = workflow.prepare_fixed_split(
            Namespace(data_root=DATA_ROOT, dataset_root=DATASET_ROOT, split_seed=42))
    workflow.validate_split(data_yaml, DATA_ROOT)
    print("LEVIR source root:", DATA_ROOT, "| fixed split:", data_yaml)
else:
    DATASET_ROOT = ROOT / "datasets"
    print("Dataset preparation disabled")

In [ ]:
# Download native pretrained family weights; NEVER use yolov8n.pt for these cases.
from ultralytics.utils.downloads import attempt_download_asset

PRETRAINED = {
    "yolov9t": (ROOT / "yolov9t.pt").resolve(),
    "yolo11n": (ROOT / "yolo11n.pt").resolve(),
}
if PREPARE_PRETRAINED:
    for family, checkpoint in PRETRAINED.items():
        if not checkpoint.is_file():
            fetched = Path(attempt_download_asset(str(checkpoint))).resolve()
            if fetched != checkpoint:
                raise RuntimeError(f"{family}: pretrained downloaded to unexpected path {fetched}")
        if not checkpoint.is_file() or checkpoint.stat().st_size == 0:
            raise FileNotFoundError(f"{family}: official pretrained checkpoint missing: {checkpoint}")
        print("Native pretrained ready:", family, checkpoint, checkpoint.stat().st_size, "bytes")

In [ ]:
# Original notebook's Linux dataloader fork workaround (required for workers > 0).
if RUN_TRAINING and WORKERS > 0:
    import multiprocessing as mp
    from ultralytics.data import build as data_build
    if "fork" not in mp.get_all_start_methods():
        raise RuntimeError("Multiprocessing fork unavailable. Choose WORKERS = 0.")
    loader = data_build.InfiniteDataLoader
    if not getattr(loader, "_ftsc_fork_context_enabled", False):
        old_init = loader.__init__
        ctx = mp.get_context("fork")
        def init_with_fork(self, *args, **kwargs):
            if kwargs.get("num_workers", 0) > 0:
                kwargs.setdefault("multiprocessing_context", ctx)
            return old_init(self, *args, **kwargs)
        loader.__init__ = init_with_fork
        loader._ftsc_fork_context_enabled = True
    print("Linux fork multiprocessing enabled for FTSC dataset workers")

In [ ]:
COMMON = [
    "--data-root", str(DATA_ROOT if DATA_ROOT is not None else Path("/marimo")),
    "--dataset-root", str(DATASET_ROOT),
    "--pretrained-v9", str(PRETRAINED["yolov9t"]),
    "--pretrained-v11", str(PRETRAINED["yolo11n"]),
    "--project", str(OUTPUT_ROOT),
    "--device", DEVICE,
    "--workers", str(WORKERS),
]
if RUN_STATIC_PREFLIGHT:
    suite.main(COMMON)
if RUN_MODEL_PREFLIGHT:
    print(suite.model_preflight(
        PRETRAINED["yolov9t"], PRETRAINED["yolo11n"]))

In [ ]:
# HF-first inventory, SHA-verified restore and fail-closed remote collision handling.
# This phase is mandatory before any RUN_TRAINING.
hf_uploader = None
hf_report = None
if RUN_HF_PREFLIGHT_AND_RESTORE or RUN_TRAINING:
    if not HF_TOKEN:
        raise RuntimeError("HF_TOKEN is mandatory for immediate seed checkpoint archival.")
    if DATA_ROOT is None or any(not p.is_file() for p in PRETRAINED.values()):
        raise RuntimeError("LEVIR dataset and both native pretrained weights must be ready.")
    if not data_yaml.is_file():
        raise RuntimeError("Frozen split YAML is missing")
    args = suite.parse_args(COMMON)
    args.data_root = DATA_ROOT.resolve()
    args.dataset_root = DATASET_ROOT.resolve()
    args.pretrained_v9 = PRETRAINED["yolov9t"]
    args.pretrained_v11 = PRETRAINED["yolo11n"]
    args.project = OUTPUT_ROOT.resolve()
    source = suite.source_preflight()
    hf_report = hf_suite.inventory(HF_TOKEN)
    print("Pinned HF results revision:", hf_report["revision"])
    print("Existing complete P345 FTSC cells:",
          [(x["case"], x["seed"]) for x in hf_report["rows"]
           if x["status"] == "MARKED_PRESENT"])
    hf_uploader = hf_suite.uploader(HF_TOKEN, HF_CACHE)
    restored = hf_suite.restore(hf_uploader, hf_report, args, data_yaml, source)
    print("Validated HF restores:", restored)

In [ ]:
# SINGLE training dispatch for only the requested new YAMLs.
# Once model+pretrained+loss preflight passes, train/evaluate/upload each seed at once.
if RUN_TRAINING:
    if hf_uploader is None or hf_report is None:
        raise RuntimeError("HF inventory/restore required before training.")
    print("Real model/loss/backward + pretrained initialization preflight:")
    print(suite.model_preflight(PRETRAINED["yolov9t"], PRETRAINED["yolo11n"]))
    for seed in SEEDS:
        for case in CASES:
            assert case in suite.CASES
            print(f"[P345 + FTSC] {case}/seed_{seed}", flush=True)
            args.resume = bool(ALLOW_RESUME)
            suite.run_one(args, case, seed, data_yaml, source)
            receipt = hf_suite.publish_or_verify(
                hf_uploader, HF_TOKEN, args, case, seed, data_yaml, source)
            print("[P345 + FTSC] HF archival:", receipt, flush=True)
    print("Selected two-family P345+FTSC runs processed.")

In [ ]:
if RUN_AGGREGATION:
    if not data_yaml.is_file():
        raise RuntimeError("Frozen LEVIR split is missing")
    suite.main(COMMON + ["--aggregate"])
    for filename in ("p345_ftsc_runs.csv", "p345_ftsc_aggregate.csv"):
        if not (OUTPUT_ROOT / filename).is_file():
            raise RuntimeError(f"Summary not created: {filename}")
    print("Six-run P345 FTSC summary:", OUTPUT_ROOT)

## Marimo Live run order

**Preflight:** Push the new codebase to GitHub, open this notebook and leave `RUN_TRAINING=False`. Check offline unit tests, static YAML graph parity, data retrieval and native checkpoints. Turn on `RUN_MODEL_PREFLIGHT=True` once to verify actual Detect strides, FTSC loss/backward and pretrained initialization.

**Pilot:** set `RUN_TRAINING=True`, `CASES=["V9T_P345_FTSC"]`, `SEEDS=[42]`. You may limit the selection to a subset of these TWO authorized cases and canonical seeds; never introduce v8 or P2/P3 cases here. The runner includes a real model preflight before GPU training, and uploads each completed seed immediately to the canonical results repository.

**Remaining five runs:** choose `CASES=["V9T_P345_FTSC", "V11N_P345_FTSC"]` and `SEEDS=[42,43,44]`. Compatible runs on HF are restored and verified, not retrained. After all six are complete, set `RUN_AGGREGATION=True`.

Output is **six seed results and two means/stds only**, not a factorial causal-effects table. The other factorial cells remain in their original distinct experimental cohorts.

**Security and provenance:** Pass HF_TOKEN (and optional GITHUB_TOKEN) as runtime environment secrets, or opt into `PROMPT_FOR_HF_TOKEN=True` for concealed entry. Never hard-code them. The source training notebook contained a literal HF token, which should be rotated. This notebook never changes the original source.